In [25]:
from typing import TypedDict,Literal 
from langchain_core.messages import HumanMessage,SystemMessage
from langchain_ollama import ChatOllama
from pydantic import BaseModel , Field
from dotenv import load_dotenv
import os 
import requests
from langchain_mcp_adapters.client import MultiServerMCPClient
import asyncio
import json
from swiggy_auth import create_oauth_provider
load_dotenv()

api_key = os.getenv("USDA_API_KEY")

class AgentState(TypedDict):
    user_query : str
    budget_amount: float | None
    food_type : str|None
    protein_target: float | None
    protein_consumed: float | None
    protein_deficit: float | None
    recommendation : str
    approval:bool
    search_results: list[dict]
    ranked_items: list[dict]
    follow_up_question: str | None
    rewrited_user_query : str|None
   
llm = ChatOllama(model = "qwen2.5:1.5b")



In [26]:
class extract(BaseModel):
    amount : float|None  = Field(description ="extract the amount from the user input default value is none")
    protein_target : float|None = Field(description = "extract the target protein from the user input default value is none")
    protein_consumed : float|None = Field(description = "extract the consumed protein from the user query default value is none")
    food_types: str|None = Field("extract the food type from the user Veg or Non veg from the user query default value is none")

system_prompt = """
You are a constraint extraction AI.

Extract the following fields from the user's query:

1. amount
   - Budget amount mentioned by the user.

2. protein_target
   - Desired protein intake in grams.

3. protein_consumed
   - Protein already consumed by the user in grams.

4. food_type
   - User's food preference.
   - Return:
     - "vegetarian" if the user explicitly mentions veg, vegetarian, pure veg.
     - "non_vegetarian" if the user explicitly mentions non veg, non-veg, chicken, fish, egg, meat, etc.
   - If not mentioned, return None.

Rules:
- Extract only information explicitly stated by the user.
- Never infer, estimate, or guess values.
- If a field is not mentioned, return None.
- If the query contains no relevant information, return None for all fields.
- Do not perform calculations.
- Do not assume that every number refers to every field.
- Do not infer vegetarian or non-vegetarian preference from budget alone.
- Standardize food_type values as:
  - vegetarian
  - non_vegetarian

Examples:

User: "I have ₹300"
amount = 300
protein_target = None
protein_consumed = None
food_type = None

User: "Need 120g protein"
amount = None
protein_target = 120
protein_consumed = None
food_type = None

User: "I've already consumed 40g protein"
amount = None
protein_target = None
protein_consumed = 40
food_type = None

User: "Veg food under 200"
amount = 200
protein_target = None
protein_consumed = None
food_type = vegetarian

User: "Non veg food under 300"
amount = 300
protein_target = None
protein_consumed = None
food_type = non_vegetarian

User: "Need 30g protein, non veg, budget 250"
amount = 250
protein_target = 30
protein_consumed = None
food_type = non_vegetarian

User: "Recommend food"
amount = None
protein_target = None
protein_consumed = None
food_type = None
"""
llm_with_structure = llm.with_structured_output(extract)

def constraint_extract(state: AgentState):
    query = state["user_query"]

    response: extract = llm_with_structure.invoke(
        [
            SystemMessage(content=system_prompt),
            HumanMessage(content=query)
        ]
    )

    return {
        "budget_amount": (
            response.amount
            if response.amount is not None
            else state.get("budget_amount")
        ),
        "protein_target": (
            response.protein_target
            if response.protein_target is not None
            else state.get("protein_target")
        ),
        "protein_consumed": (
            response.protein_consumed
            if response.protein_consumed is not None
            else state.get("protein_consumed")
        ),
        "food_type": (
        response.food_type
        if response.food_type is not None
        else state.get("food_type")
        )
    }


In [27]:

class followup(BaseModel):

    question : str
system_prompt2 = """
You are a follow-up question generation assistant.

Your task is to generate a natural question asking the user only for the missing information.

Possible missing information:
- budget → user's spending limit
- protein_target → desired protein intake in grams
- food_type → vegetarian or non-vegetarian preference

Rules:
- Ask only for the missing information provided.
- Do not ask for information that is already available.
- If multiple pieces of information are missing, combine them into a single concise question.
- Use natural, conversational language.
- For food_type, ask whether the user prefers vegetarian or non-vegetarian food.
- For protein_target, ask how much protein the user wants.
- For budget, ask for the user's budget.
- Keep the question short and clear.
- Do not explain your reasoning.
- Do not return anything except the question.
"""

def Missing_constraint(state:AgentState):
    missing = []

    if state["budget_amount"] is None:
        missing.append("budget")

    if state["protein_target"] is None:
        missing.append("protein_target")
    if state['food_type'] is None:
        missing.append("food_type")
    if not missing:
        return {
            "missing_constraints": [],
            "follow_up_question": None
        }
    
    llm_with_structure2 = llm.with_structured_output(followup)
    
    response: followup = llm_with_structure2.invoke(
        [
            SystemMessage(content=system_prompt2),
               HumanMessage(
    content=f"""
    The user has not provided the following information:

    {', '.join(missing)}

    Generate a follow-up question asking only for this information.
"""
)
            ])
        

    return {
        "follow_up_question": response.question
    }


In [28]:
oauth_provider = create_oauth_provider("https://mcp.swiggy.com/food")

client = MultiServerMCPClient({
    "swiggy-food": {
        "url": "https://mcp.swiggy.com/food",
        "transport": "streamable_http",
        "auth": oauth_provider,
    },
})

tools = await client.get_tools()

In [29]:
tools

[StructuredTool(name='get_addresses', description='Swiggy (Instamart/Food): Get all saved delivery addresses for the authenticated Swiggy user, sorted by last order date. This tool works for Swiggy Instamart and Food services. Addresses are returned WITHOUT coordinates (latitude/longitude) for privacy protection. No parameters needed - authentication is handled automatically.\n\n📍 IMPORTANT — STOP here and let the user choose:\n1. Show the address list to the user\n2. Ask: "Which address would you like to use for delivery?"\n3. Do NOT call any other tool until the user has selected an address\n4. Remember the selected addressId for all subsequent operations\n5. If no addresses are returned, inform the user that they need to add an address first', args_schema={'type': 'object', 'properties': {}, 'required': []}, metadata={'title': 'Get Delivery Addresses', 'readOnlyHint': True, 'destructiveHint': False, 'idempotentHint': True, 'openWorldHint': False, '_meta': {'ui': {'resourceUri': 'ui:

In [7]:
get_addresses = next(t for t in tools if t.name == "get_addresses")
get_restaurant = next(t for t in tools if t.name == "search_restaurants")
get_restaurant_menu = next(t for t in tools if t.name == "get_restaurant_menu")
search_menu = next(t for t in tools if t.name == "search_menu" )

In [ ]:
llm_search = ChatOllama(model = "qwen2.5:7b")
class search_query(BaseModel):
    search_query:str = Field(description= "store only the generated query from the llm default None")


system_prompt3 = """
You are a restaurant discovery query generator.

Your task is to generate ONE broad food category or restaurant-search term that will help a restaurant search engine find relevant restaurants.

IMPORTANT:
The search engine works best with broad food categories, not specific dishes.

Return exactly ONE search term.

Rules:

1. Return exactly one search term.

2. Return only English text.

3. Do not return multiple terms.

4. Do not return comma-separated lists.

5. Do not return complete sentences.

6. Do not return restaurant names.

7. Do not return budgets.

Never return:
- under 300
- budget friendly
- cheap
- affordable

8. Do not return dietary labels.

Never return:
- vegetarian
- non vegetarian
- veg
- non veg
- healthy
- protein rich
- high protein

9. Prefer broad food categories that are commonly used to discover restaurants.

Good outputs:
- biryani
- rolls
- dosa
- thali
- pizza
- burger
- chinese
- kebabs

Bad outputs:
- chicken biryani
- paneer tikka
- egg fried rice
- grilled chicken
- vegetarian
- non vegetarian
- budget friendly

Guidelines:

If the user wants vegetarian food:
Choose one category such as:
- thali
- dosa
- paneer

If the user wants non-vegetarian food:
Choose one category such as:
- biryani
- rolls
- kebabs
- chicken

If the user wants breakfast:
Prefer:
- dosa
- idli

If the user wants snacks:
Prefer:
- rolls
- burger

Return only the search term.
"""

llm_with_structure3 = llm_search.with_structured_output(search_query)


async def food_search(state:AgentState):
    budget = state["budget_amount"]
    query = state["user_query"]
    addresses_response = await get_addresses.ainvoke({})

    addresses = json.loads(addresses_response[0]["text"])
    question = "Which address would you like to use?"
    for i , address in enumerate(addresses , start = 1):
          question+=f"\n{i}. {address['label']}"
    address_id = None
    for address in addresses:
        if address["label"].lower() == query.lower():
            address_id = address["id"]
            break
       
    if address_id is None:
        return {
        "follow_up_question": question
        }   
    restaurant_query : search_query = llm_with_structure3.invoke([SystemMessage(content = system_prompt3), HumanMessage(content = query)])

    restaurants = await get_restaurant.ainvoke({"query": restaurant_query, "addressId" : address_id})
    
    for res in restaurants:
        res_id = res["ID"]
    menu  = await get_restaurant_menu.ainvoke({
        "addressId" : address_id,
        "restaurantId" : res_id
    })


In [51]:
get_restaurant_menu.args_schema

{'type': 'object',
 'properties': {'addressId': {'type': 'string',
   'description': 'Address ID from get_addresses tool'},
  'restaurantId': {'type': 'string',
   'description': 'Restaurant ID to fetch menu for (from search_restaurants)'},
  'page': {'type': 'number',
   'description': 'Page number for pagination (default: 1)',
   'default': 1},
  'pageSize': {'type': 'number',
   'description': 'Number of categories per page (default: 5, max: 8)',
   'default': 5}},
 'required': ['addressId', 'restaurantId']}

In [ ]:
restaurants = await get_restaurant.ainvoke({"query": "kebabs",
    "addressId": "d7dj5lfj7k7gunt915h0"})

print(type(restaurants))
print(restaurants)

<class 'list'>
[{'type': 'text', 'text': 'Found 10 restaurants for "kebabs":\n1. Barbeque Nation (Ad) — North Indian, Barbecue, Kebabs, Biryani, Street Food, Snacks | 4★ | 39 min | ₹300 for two (ID: 1152842)\n2. Mio Amore - The Cake Shop — Bakery, Desserts, Snacks, Confectionery | 4.5★ | 15 min | ₹150 for two (ID: 391900)\n3. Karim\'s- Original From Jama Masjid Delhi-6 (Ad) — Biryani, Kebabs | 4.3★ | 37 min | ₹850 for two (ID: 985483)\n4. Let\'s Eat — Biryani, Chinese, North Indian, Mughlai | 4.5★ | 36 min | ₹300 for two (ID: 189007)\n5. Royal\'S (Ad) — Indian, Chinese, Biryani | 4.3★ | 48 min | ₹200 for two (ID: 319114)\n6. Panchu Gopal Dey\'s Food Plaza — Biryani, Chinese, Indian, South Indian, Tandoor | 4.5★ | 29 min | ₹250 for two (ID: 209751)\n7. Dhakeswari Chinese Restaurant (Ad) — Chinese, Bengali, Biryani | 4.4★ | 36 min | ₹400 for two (ID: 630821)\n8. Royal Hut — Bengali, Mughlai | 4.4★ | 37 min | ₹550 for two (ID: 304225)\n9. Alahie\' Family Restaurant (Ad) — Biryani, Chinese

In [55]:
menu  = await get_restaurant_menu.ainvoke({
        "addressId" : "d7dj5lfj7k7gunt915h0",
        "restaurantId" : 1152842
    })
menu

[{'type': 'text',
  'text': 'Menu for Barbeque Nation (ID: 1152842) [image: https://media-assets.swiggy.com/swiggy/image/upload/RX_THUMBNAIL/IMAGES/VENDOR/2025/7/22/bfb0f48e-5c04-469a-a2ed-f988b6b3b4f9_1152842.jpg]\nPage 1 — 5 of 12 categories. Use page=2 for more.\n\n## Items starting at 159\n  - Tandoori Chicken Wings (6 Pcs) — ₹199 | Non-veg, Bestseller, has addons [image: https://media-assets.swiggy.com/swiggy/image/upload/FOOD_CATALOG/IMAGES/CMS/2025/11/19/e1eac873-9f3a-4ce5-a576-068b25bef6bb_26ef2c57-df5f-456b-abb8-a227a9c1877c.jpg] (ID: 186894296)\n  - Hariyali Paneer Tikka (8 Pcs) — ₹349 | Veg [image: https://media-assets.swiggy.com/swiggy/image/upload/FOOD_CATALOG/IMAGES/CMS/2025/7/31/07846d2a-93d8-4ee8-b9a4-ed868db5642e_b366dfdf-fc31-4a14-9688-c982387b0f11.jpg] (ID: 193031661)\n  - Aachari Paneer Tikka (8 Pcs) — ₹389 | Veg, has addons [image: https://media-assets.swiggy.com/swiggy/image/upload/FOOD_CATALOG/IMAGES/CMS/2025/7/31/cb46d31c-b61b-4c0c-93fb-b482dee35e34_f0ffd5d1-e43

In [48]:
restaurant_query : search_query = llm_with_structure3.invoke([SystemMessage(content = system_prompt3), HumanMessage(content = "Need 60g protein, non-vegetarian, budget ₹250")])
restaurant_query

search_query(search_query='kebabs')

In [56]:
print(restaurants[0]["text"])

Found 10 restaurants for "kebabs":
1. Barbeque Nation (Ad) — North Indian, Barbecue, Kebabs, Biryani, Street Food, Snacks | 4★ | 39 min | ₹300 for two (ID: 1152842)
2. Mio Amore - The Cake Shop — Bakery, Desserts, Snacks, Confectionery | 4.5★ | 15 min | ₹150 for two (ID: 391900)
3. Karim's- Original From Jama Masjid Delhi-6 (Ad) — Biryani, Kebabs | 4.3★ | 37 min | ₹850 for two (ID: 985483)
4. Let's Eat — Biryani, Chinese, North Indian, Mughlai | 4.5★ | 36 min | ₹300 for two (ID: 189007)
5. Royal'S (Ad) — Indian, Chinese, Biryani | 4.3★ | 48 min | ₹200 for two (ID: 319114)
6. Panchu Gopal Dey's Food Plaza — Biryani, Chinese, Indian, South Indian, Tandoor | 4.5★ | 29 min | ₹250 for two (ID: 209751)
7. Dhakeswari Chinese Restaurant (Ad) — Chinese, Bengali, Biryani | 4.4★ | 36 min | ₹400 for two (ID: 630821)
8. Royal Hut — Bengali, Mughlai | 4.4★ | 37 min | ₹550 for two (ID: 304225)
9. Alahie' Family Restaurant (Ad) — Biryani, Chinese, Snacks, Tandoor, North Indian | 4★ | 52 min | ₹200 for